# Medici Price Prediction - Data Exploration & First Model

This notebook walks through:
1. Connecting to Azure SQL DB and discovering the schema
2. Loading and exploring pricing data
3. Feature engineering
4. Training a forecasting model with Darts
5. Generating pricing recommendations

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv

load_dotenv('../.env')
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print('Ready')

## 1. Connect to Azure SQL & Discover Schema

In [ ]:
from src.data.db_loader import discover_schema, get_engine

# Discover what tables exist in your Azure SQL DB
schema = discover_schema()
for table, cols in schema.items():
    print(f'\n=== {table} ===')
    for col in cols:
        print(f'  - {col}')

## 2. Load Pricing Data

Adjust the table name and columns below to match your actual DB schema.
After running `discover_schema()` above, you'll know what's available.

In [ ]:
from src.data.db_loader import load_table, load_daily_pricing

# Option A: Load from a specific table
# df = load_table('your_table_name', limit=1000)

# Option B: Load daily pricing data
# df = load_daily_pricing(start_date='2024-01-01')

# Option C: For testing without DB, generate synthetic data
np.random.seed(42)
dates = pd.date_range('2024-01-01', '2025-12-31', freq='D')
base_price = 500

# Simulate seasonal patterns + noise
seasonal = 100 * np.sin(2 * np.pi * np.arange(len(dates)) / 365)  # yearly cycle
weekly = 50 * np.sin(2 * np.pi * np.arange(len(dates)) / 7)  # weekly cycle
trend = np.linspace(0, 50, len(dates))  # slight upward trend
noise = np.random.normal(0, 30, len(dates))

df = pd.DataFrame({
    'date': dates,
    'price': base_price + seasonal + weekly + trend + noise,
    'occupancy_rate': np.clip(0.5 + 0.2 * np.sin(2 * np.pi * np.arange(len(dates)) / 365) + np.random.normal(0, 0.1, len(dates)), 0, 1),
    'competitor_avg_price': base_price + seasonal * 0.8 + np.random.normal(0, 40, len(dates)),
})

print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
# Basic statistics
df.describe()

In [ ]:
# Visualize price over time
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df['date'], df['price'], linewidth=0.8)
axes[0].set_ylabel('Price (ILS)')
axes[0].set_title('Daily Room Price')

axes[1].plot(df['date'], df['occupancy_rate'], color='green', linewidth=0.8)
axes[1].set_ylabel('Occupancy Rate')
axes[1].set_title('Occupancy')

axes[2].plot(df['date'], df['price'], label='Our Price', linewidth=0.8)
axes[2].plot(df['date'], df['competitor_avg_price'], label='Competitor Avg', linewidth=0.8, alpha=0.7)
axes[2].set_ylabel('Price (ILS)')
axes[2].set_title('Price vs Competitors')
axes[2].legend()

plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
from src.features.engineering import prepare_features

df_features = prepare_features(df, date_col='date', price_col='price')
print(f'Features added: {len(df_features.columns) - len(df.columns)} new columns')
print(f'Total columns: {list(df_features.columns)}')
df_features.head()

In [ ]:
# Correlation with price
numeric_cols = df_features.select_dtypes(include=[np.number]).columns
correlations = df_features[numeric_cols].corr()['price'].sort_values(ascending=False)
print('Top correlations with price:')
print(correlations.head(15))

## 4. Train Forecasting Model

In [ ]:
from src.models.forecaster import HotelPriceForecaster, compare_models

# Compare multiple models
comparison = compare_models(df, date_col='date', target_col='price')
print('Model Comparison:')
comparison

In [ ]:
# Train the best model and predict
forecaster = HotelPriceForecaster(model_name='lightgbm', horizon=30)
metrics = forecaster.train(df, date_col='date', target_col='price')
print(f'Training metrics: {metrics}')

predictions = forecaster.predict(30)
print(f'\nPredictions for next 30 days:')
predictions.head(10)

In [ ]:
# Visualize predictions
fig, ax = plt.subplots(figsize=(14, 5))

# Last 90 days of actual data
recent = df.tail(90)
ax.plot(recent['date'], recent['price'], label='Actual', color='blue')
ax.plot(predictions['date'], predictions['predicted_price'], label='Predicted', color='red', linestyle='--')

ax.set_xlabel('Date')
ax.set_ylabel('Price (ILS)')
ax.set_title('Price Forecast - Next 30 Days')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Dynamic Pricing Recommendations

In [ ]:
from src.models.pricing import DynamicPricer

pricer = DynamicPricer(min_price=200, max_price=2000)

# Generate pricing table
pricing_table = pricer.generate_pricing_table(predictions)
print('Pricing Recommendations:')
pricing_table.head(10)

In [ ]:
# Visualize recommended vs predicted
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(pricing_table['date'], pricing_table['predicted_price'], label='ML Predicted', marker='o', markersize=3)
ax.plot(pricing_table['date'], pricing_table['recommended_price'], label='Recommended', marker='s', markersize=3)
ax.fill_between(pricing_table['date'], pricing_table['predicted_price'], pricing_table['recommended_price'], alpha=0.2)
ax.set_xlabel('Date')
ax.set_ylabel('Price (ILS)')
ax.set_title('Predicted vs Recommended Pricing')
ax.legend()
plt.tight_layout()
plt.show()

## Next Steps

1. **Connect real data** - Set DATABASE_URL in .env and replace synthetic data with Azure SQL data
2. **Tune models** - Experiment with hyperparameters and feature selection
3. **Add covariates** - Use occupancy, competitors, events as model inputs
4. **Deploy API** - Run `python -m src.api.main` to start the prediction server
5. **Monitor** - Track prediction accuracy over time